In [1]:
import pandas as pd

file = "mobile_shoppee_curation_project.xlsx"

sheet_names = pd.ExcelFile(file).sheet_names
print(sheet_names)

['Customers', 'MobileModels', 'ShopLocations', 'Sales', 'Stocks', 'Offers', 'DataDictionary']


In [3]:
import pandas as pd
sales = pd.read_csv(r"C:\Users\TANU\Documents\EDA/Sales.csv")
models = pd.read_csv(r"C:\Users\TANU\Documents\EDA/MobileModels.csv")

In [4]:
#Step 6: Data Integration & Merging

In [5]:
#1.Merging dataset

In [6]:
import pandas as pd

folder = r"C:\Users\TANU\Documents\EDA"

# Load all datasets
customers = pd.read_csv(folder + "\\Customers.csv")
models    = pd.read_csv(folder + "\\MobileModels.csv")
shops     = pd.read_csv(folder + "\\ShopLocations.csv")
sales     = pd.read_csv(folder + "\\Sales.csv")
offers    = pd.read_csv(folder + "\\Offers.csv")
stocks    = pd.read_csv(folder + "\\Stocks.csv")

# 1) FIX ORPHAN KEYS (so merges don't produce nulls)
sales["customer_id"] = sales["customer_id"].where(
    sales["customer_id"].isin(customers["customer_id"]), "UNKNOWN_CUST"
)

sales["model_id"] = sales["model_id"].where(
    sales["model_id"].isin(models["model_id"]), "UNKNOWN_MODEL"
)

sales["shop_id"] = sales["shop_id"].where(
    sales["shop_id"].isin(shops["shop_id"]), "UNKNOWN_SHOP"
)

sales["offer_id"] = sales.get("offer_id", pd.Series(["NO_OFFER"] * len(sales)))

# 2) MERGE IN CORRECT ORDER

# Step A — Sales + Customers
unified = sales.merge(customers, on="customer_id", how="left")

# Step B — Add model details
unified = unified.merge(
    models,
    on="model_id",
    how="left",
    suffixes=("", "_model")
)

# Step C — Add shop details
unified = unified.merge(
    shops.rename(columns={"location": "shop_location"}),
    on="shop_id",
    how="left",
    suffixes=("", "_shop")
)

# Step D — Add offer details
unified = unified.merge(
    offers,
    on=["model_id", "shop_id"],
    how="left",
    suffixes=("", "_offer")
)

# Step E — Add stock availability
unified = unified.merge(
    stocks[["shop_id", "model_id", "available_units"]],
    on=["shop_id", "model_id"],
    how="left"
)

# 3) SAVE FINAL FILE
output_path = folder + "\\Unified_Schema.csv"
unified.to_csv(output_path, index=False)

print("\n✔ Unified schema created successfully!")
print(f"Saved to: {output_path}")
print(f"Final shape: {unified.shape}")

print("\nSample rows:")
print(unified.head())



✔ Unified schema created successfully!
Saved to: C:\Users\TANU\Documents\EDA\Unified_Schema.csv
Final shape: (42665, 35)

Sample rows:
    sale_id customer_id model_id   shop_id   sale_date  quantity  discount  \
0  SALE4000    CUST2083  MOD2111  SHOP3068  2024-11-26         1      1.05   
1  SALE4001    CUST2384  MOD2021  SHOP3063  2021-07-13         1     10.48   
2  SALE4002    CUST6189  MOD2008  SHOP3004  2025-05-04         2     13.55   
3  SALE4003    CUST4992  MOD2105  SHOP3041  2023-02-28         1      7.57   
4  SALE4004    CUST4707  MOD2109  SHOP3024  2020-04-12         1     18.79   

   final_price payment_mode     status  ...    price shop_location  \
0   22316.1935  Credit Card  Delivered  ...  22553.0          Pune   
1   26218.6176         Cash    Pending  ...  29288.0          Pune   
2   36469.7970         Cash  Cancelled  ...  21093.0         Delhi   
3   37230.8040  Credit Card    Pending  ...  40280.0       Kolkata   
4   35201.2866          UPI  Delivered  ...  

In [7]:
#2.Orphan key resolution code:
import pandas as pd

folder = r"C:\Users\TANU\Documents\EDA"

# Load all datasets
customers = pd.read_csv(folder + "\\Customers.csv")
models = pd.read_csv(folder + "\\MobileModels.csv")
shops = pd.read_csv(folder + "\\ShopLocations.csv")
sales = pd.read_csv(folder + "\\Sales.csv")
offers = pd.read_csv(folder + "\\Offers.csv")
stocks = pd.read_csv(folder + "\\Stocks.csv")

print("Files loaded.")
# FIX ORPHAN KEYS IN SALES

# Replace invalid customer_id with safe dummy ID
sales["customer_id"] = sales["customer_id"].where(
    sales["customer_id"].isin(customers["customer_id"]),
    "CUST_UNKNOWN"
)

# Replace invalid model_id with safe dummy ID
sales["model_id"] = sales["model_id"].where(
    sales["model_id"].isin(models["model_id"]),
    "MOD_UNKNOWN"
)

# Replace invalid shop_id with safe dummy ID
sales["shop_id"] = sales["shop_id"].where(
    sales["shop_id"].isin(shops["shop_id"]),
    "SHOP_UNKNOWN"
)

# Replace invalid offer_id with safe dummy ID
if "offer_id" in sales.columns:
    sales["offer_id"] = sales["offer_id"].where(
        sales["offer_id"].isin(offers["offer_id"]),
        "OFF_UNKNOWN"
    )
else:
    # Sales had no offer_id → create one
    sales["offer_id"] = "OFF_UNKNOWN"
    
#FIX ORPHAN KEYS IN OFFERS

offers["model_id"] = offers["model_id"].where(
    offers["model_id"].isin(models["model_id"]),
    "MOD_UNKNOWN"
)

offers["shop_id"] = offers["shop_id"].where(
    offers["shop_id"].isin(shops["shop_id"]),
    "SHOP_UNKNOWN"
)

# FIX ORPHAN KEYS IN STOCKS

stocks["model_id"] = stocks["model_id"].where(
    stocks["model_id"].isin(models["model_id"]),
    "MOD_UNKNOWN"
)

stocks["shop_id"] = stocks["shop_id"].where(
    stocks["shop_id"].isin(shops["shop_id"]),
    "SHOP_UNKNOWN"
)

# SAVE FIXED FILES
sales.to_csv(folder + "\\Sales.csv", index=False)
offers.to_csv(folder + "\\Offers.csv", index=False)
stocks.to_csv(folder + "\\Stocks.csv", index=False)

print("\n✔ ORPHAN KEYS FIXED FOR:")
print("   - Sales")
print("   - Offers")
print("   - Stocks")

print("\n✔ Dummy IDs used:")
print("   CUST_UNKNOWN")
print("   MOD_UNKNOWN")
print("   SHOP_UNKNOWN")
print("   OFF_UNKNOWN")


Files loaded.

✔ ORPHAN KEYS FIXED FOR:
   - Sales
   - Offers
   - Stocks

✔ Dummy IDs used:
   CUST_UNKNOWN
   MOD_UNKNOWN
   SHOP_UNKNOWN
   OFF_UNKNOWN


In [8]:
#Loading cleaned file:
import pandas as pd

folder = r"C:\Users\TANU\Documents\EDA"

# Load cleaned files
customers = pd.read_csv(folder + "\\Customers.csv")
models = pd.read_csv(folder + "\\MobileModels.csv")
shops = pd.read_csv(folder + "\\ShopLocations.csv")
sales = pd.read_csv(folder + "\\Sales.csv")
offers = pd.read_csv(folder + "\\Offers.csv")
stocks = pd.read_csv(folder + "\\Stocks.csv")

print("All cleaned files loaded.")

# FIX: MobileModels uses column name  "price"
# So rename it to base_price BEFORE merging
models = models.rename(columns={"price": "base_price"})

model_cols = ["model_id", "brand", "model_name", "ram_gb", "storage_gb", "base_price"]

# STEP 1 — MERGE SALES + CUSTOMERS
unified = sales.merge(
    customers,
    on="customer_id",
    how="left"
)

# STEP 2 — MERGE MODELS
unified = unified.merge(
    models[model_cols],
    on="model_id",
    how="left"
)

# STEP 3 — MERGE SHOPS
shops = shops.rename(columns={"location": "shop_location"})

unified = unified.merge(
    shops[["shop_id", "shop_location"]],
    on="shop_id",
    how="left"
)

# STEP 4 — MERGE OFFERS
unified = unified.merge(
    offers[["offer_id", "discount_percent", "start_date", "end_date", "campaign_name"]],
    on="offer_id",
    how="left"
)

# STEP 5 — MERGE STOCKS
unified = unified.merge(
    stocks[["model_id", "shop_id", "available_units"]],
    on=["model_id", "shop_id"],
    how="left"
)

# STEP 6 — SAVE UNIFIED SCHEMA
save_path = folder + "\\Unified_Schema.csv"
unified.to_csv(save_path, index=False)

print("\n✔ Unified schema created and saved!")
print("Path:", save_path)

print("\nUnified shape:", unified.shape)
print("\nSample rows:\n")
print(unified.head())


All cleaned files loaded.

✔ Unified schema created and saved!
Path: C:\Users\TANU\Documents\EDA\Unified_Schema.csv

Unified shape: (42659, 32)

Sample rows:

    sale_id customer_id model_id   shop_id   sale_date  quantity  discount  \
0  SALE4000    CUST2083  MOD2111  SHOP3068  2024-11-26         1      1.05   
1  SALE4001    CUST2384  MOD2021  SHOP3063  2021-07-13         1     10.48   
2  SALE4002    CUST6189  MOD2008  SHOP3004  2025-05-04         2     13.55   
3  SALE4003    CUST4992  MOD2105  SHOP3041  2023-02-28         1      7.57   
4  SALE4004    CUST4707  MOD2109  SHOP3024  2020-04-12         1     18.79   

   final_price payment_mode     status  ... model_name_y ram_gb_y  \
0   22316.1935  Credit Card  Delivered  ...      Series6        4   
1   26218.6176         Cash    Pending  ...     Series16        4   
2   36469.7970         Cash  Cancelled  ...     Series19        6   
3   37230.8040  Credit Card    Pending  ...      Series2       12   
4   35201.2866          UPI

In [9]:
import pandas as pd

folder = r"C:\Users\TANU\Documents\EDA"
unified = pd.read_csv(folder + "\\Unified_Schema.csv")

# DIMENSION TABLES

# ---------------- DimCustomer ----------------
DimCustomer = unified[[
    "customer_id", "full_name", "email", "phone", "city", "loyalty_points"
]].drop_duplicates().reset_index(drop=True)

# ---------------- DimModel ----------------
DimModel = unified[[
    "model_id",
    "brand_x", "model_name_x", "ram_gb_x", "storage_gb_x", "base_price_x"
]].drop_duplicates().reset_index(drop=True)

# Rename cleanly
DimModel = DimModel.rename(columns={
    "brand_x": "brand",
    "model_name_x": "model_name",
    "ram_gb_x": "ram_gb",
    "storage_gb_x": "storage_gb",
    "base_price_x": "base_price"
})

# ---------------- DimShop ----------------
DimShop = unified[[
    "shop_id", "shop_location"
]].drop_duplicates().reset_index(drop=True)

# ---------------- DimOffer ----------------
DimOffer = unified[[
    "offer_id", "discount_percent", "start_date", "end_date", "campaign_name"
]].drop_duplicates().reset_index(drop=True)

# ---------------- DimStock ----------------
DimStock = unified[[
    "shop_id", "model_id", "available_units"
]].drop_duplicates().reset_index(drop=True)

# FACT TABLE (FACTSALES)

FactSales = unified[[
    "sale_id", "sale_date", "customer_id", "model_id", "shop_id",
    "offer_id", "quantity", "discount", "final_price",
    "payment_mode", "status"
]].copy()

# SAVE ALL TABLES

DimCustomer.to_csv(folder + "\\DimCustomer.csv", index=False)
DimModel.to_csv(folder + "\\DimModel.csv", index=False)
DimShop.to_csv(folder + "\\DimShop.csv", index=False)
DimOffer.to_csv(folder + "\\DimOffer.csv", index=False)
DimStock.to_csv(folder + "\\DimStock.csv", index=False)
FactSales.to_csv(folder + "\\FactSales.csv", index=False)

print("Star Schema Created Successfully!")
print("Files saved:")
print("✓ DimCustomer.csv")
print("✓ DimModel.csv")
print("✓ DimShop.csv")
print("✓ DimOffer.csv")
print("✓ DimStock.csv")
print("✓ FactSales.csv")


Star Schema Created Successfully!
Files saved:
✓ DimCustomer.csv
✓ DimModel.csv
✓ DimShop.csv
✓ DimOffer.csv
✓ DimStock.csv
✓ FactSales.csv
